# Modifying 1000G data to exemplify IDEAL-GENOM usage

In this notebook we will modify the 1000G dataset by adding synthetic phenotype data to demonstrate the capabilities of the IDEAL-GENOM library. The original 1000 Genomes dataset does not include phenotype information, so we will randomly assign case/control status to enable quality control and GWAS analysis examples.

Firstly, let us import the required libraries.

In [1]:
import sys
import os
import shutil

from pathlib import Path

import pandas as pd
import numpy as np


Library path added to sys.path: /home/luis/CGE/ideal-genom-qc


Now, let us add the library path to PATH.

In [ ]:
# add parent directory to path
library_path = os.path.abspath('..')
if library_path not in sys.path:
    sys.path.append(library_path)

library_path = Path(library_path)

print(f"Library path added to sys.path: {library_path}")

Then, we can import a particular function from the library to run PLINK2 commands.

In [2]:
from ideal_genom.core.executor import run_plink2

Let us set up the directory structure for our test data. We'll create paths for:
- `inputData`: Original and processed genetic data files
- `outputData`: Analysis results and quality control outputs
- `config`: Configuration files for the pipeline

In [3]:
DATA_PATH = library_path / 'ideal_genom' / 'data'

test_data = DATA_PATH / 'test_data'
if not test_data.exists():
    test_data.mkdir(parents=True, exist_ok=True)

In [ ]:
inputData = test_data / 'inputData'
outputData = test_data / 'outputData'
config    = test_data / 'config'

In the next step we check if the updated 1000G files with phenotypes already exist (.bim, .fam, .bed files). If they don't exist, we copy the original 1000G data and create the directory structure. This avoids unnecessary data duplication if the notebook is run multiple times.

In [ ]:
updated_pheno = (inputData / '1kG_phase3_GRCh38_updated.bim').exists() and (inputData / '1kG_phase3_GRCh38_updated.fam').exists() and (inputData / '1kG_phase3_GRCh38_updated.bed').exists()

if not updated_pheno:
    (test_data / 'inputData').mkdir(parents=True, exist_ok=True)
    (test_data / 'outputData').mkdir(parents=True, exist_ok=True)
    (test_data / 'config').mkdir(parents=True, exist_ok=True)

    source_dir = DATA_PATH / '1000genomes_build_38'
    shutil.copytree(source_dir, test_data, dirs_exist_ok=True)

Since the 1000G dataset does not have phenotypes, we are going to randomly assign binary phenotypes to each sample. We'll use:
- **1 = Control** (unaffected)
- **2 = Case** (affected)

This phenotype information will be used later during variant QC and GWAS examples. We set a random seed (42) to ensure reproducibility of the phenotype assignments.

In [ ]:
df_1kg_fam = pd.read_csv(
    test_data / '1kG_phase3_GRCh38.fam',
    sep=r'\s+',
    header=None,
    names=["FID", "IID", "PAT", "MAT", "SEX", "PHENO"],
    engine='python'
)
df_1kg_fam.head()

,FID,IID,PAT,MAT,SEX,PHENO
0,0,HG00096,0,0,1,-9
1,0,HG00097,0,0,2,-9
2,0,HG00099,0,0,2,-9
3,0,HG00100,0,0,2,-9
4,0,HG00101,0,0,1,-9


In [ ]:
np.random.seed(42)
df_1kg_fam["PHENO"] = np.random.choice([1, 2], size=len(df_1kg_fam))

Now we save the updated phenotype data to a new .fam file. This file will be used by PLINK2 to update the original dataset with our synthetic phenotypes.

In [ ]:
df_1kg_fam.to_csv(
    test_data / '1kG_phase3_GRCh38_to_update.fam',
    sep=' ',
    header=False,
    index=False
)

Next, we use PLINK2 to create a new binary fileset (.bed/.bim/.fam) that incorporates our updated phenotype information. The `--make-bed` command generates the binary format, and the `--fam` flag specifies our custom .fam file with the new phenotypes.

In [9]:
plink2_args = [
            "--bfile", str(test_data / '1kG_phase3_GRCh38'),
            "--make-bed",
            "--out", str(inputData / '1kG_phase3_GRCh38_updated'),
            "--fam", str(test_data / '1kG_phase3_GRCh38_to_update.fam')
        ]

run_plink2(plink2_args)

PLINK v2.0.0-a.6.26LM AVX2 Intel (26 Oct 2025)      cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang    GNU General Public License v3
Logging to /home/luis/CGE/ideal-genom-qc/ideal_genom/data/test_data/inputData/1kG_phase3_GRCh38_updated.log.
Options in effect:
  --bfile /home/luis/CGE/ideal-genom-qc/ideal_genom/data/test_data/1kG_phase3_GRCh38
  --make-bed
  --out /home/luis/CGE/ideal-genom-qc/ideal_genom/data/test_data/inputData/1kG_phase3_GRCh38_updated
  --psam /home/luis/CGE/ideal-genom-qc/ideal_genom/data/test_data/1kG_phase3_GRCh38_to_update.fam

Start time: Mon Jan 19 14:32:28 2026
63863 MiB RAM detected, ~56479 available; reserving 31931 MiB for main
workspace.
Using up to 32 threads (change this with --threads).
3202 samples (1603 females, 1598 males, 1 ambiguous; 2583 founders) loaded from
/home/luis/CGE/ideal-genom-qc/ideal_genom/data/test_data/1kG_phase3_GRCh38_to_update.fam.
64068184 variants loaded from
/home/luis/CGE/ideal-genom-qc/ideal_genom/

CompletedProcess(args=['plink2', '--bfile', '/home/luis/CGE/ideal-genom-qc/ideal_genom/data/test_data/1kG_phase3_GRCh38', '--make-bed', '--out', '/home/luis/CGE/ideal-genom-qc/ideal_genom/data/test_data/inputData/1kG_phase3_GRCh38_updated', '--fam', '/home/luis/CGE/ideal-genom-qc/ideal_genom/data/test_data/1kG_phase3_GRCh38_to_update.fam'], returncode=0)

Finally, we clean up intermediate files to save disk space. This removes the temporary 1000G files from the test_data directory that were used during processing. The updated files with phenotypes are safely stored in the `inputData` subdirectory.

In [11]:
lst = test_data.glob('1kG_phase3_GRCh38*')
for file in lst:
    file.unlink()

/home/luis/CGE/ideal-genom-qc/ideal_genom/data/test_data/1kG_phase3_GRCh38.psam
/home/luis/CGE/ideal-genom-qc/ideal_genom/data/test_data/1kG_phase3_GRCh38.bed
/home/luis/CGE/ideal-genom-qc/ideal_genom/data/test_data/1kG_phase3_GRCh38.log
/home/luis/CGE/ideal-genom-qc/ideal_genom/data/test_data/1kG_phase3_GRCh38.fam
/home/luis/CGE/ideal-genom-qc/ideal_genom/data/test_data/1kG_phase3_GRCh38_to_update.fam
/home/luis/CGE/ideal-genom-qc/ideal_genom/data/test_data/1kG_phase3_GRCh38.bim
